# Lab 4 · Repaired corrective retrieval foundation

## Goal
Preserve evidence identity through a bounded retrieval, repair, and answer loop.

## Setup
Use the Labs 3–8 environment. Implement the tasks below, set RUN_EXERCISES=True, restart and run all. Default execution does not solve or grade the assignment. All fixtures are invented. Instructor solutions are distributed separately.

In [1]:
RUN_EXERCISES = False

## Provided support
These utilities support the assignment; the agent/retrieval logic is your work.

In [2]:
from __future__ import annotations

import ast

import math

import operator

import re

from dataclasses import dataclass, field

from heapq import nlargest

def normalize_search(response,max_urls=2,max_chars=15_000):
    """Preserve URL provenance, cap records, and reject malformed records."""
    if type(max_urls) is not int or not 1<=max_urls<=10:raise ValueError("max_urls must be 1..10")
    if type(max_chars) is not int or not 1<=max_chars<=15_000:raise ValueError("max_chars must be 1..15000")
    rows=response.get("results",[]) if isinstance(response,dict) else []
    if not isinstance(rows,list):return []
    out=[];seen=set()
    for row in rows:
        if not isinstance(row,dict):continue
        raw,url=row.get("raw_content"),row.get("url")
        if not isinstance(raw,str) or not raw.strip() or not isinstance(url,str) or not url.startswith(("https://","http://")) or url in seen:continue
        seen.add(url);out.append({"id":url,"source":url,"text":raw[:max_chars]})
        if len(out)==max_urls:break
    return out

def search_and_scrape(query,api_key,client_factory,max_urls=2):
    if not isinstance(query,str) or not query.strip() or len(query)>1000:raise ValueError("invalid query")
    if not isinstance(api_key,str) or not api_key:raise ValueError("missing key")
    if type(max_urls) is not int or not 1<=max_urls<=10:raise ValueError("max_urls must be 1..10")
    client=client_factory(api_key=api_key)
    response=client.search(query=query,max_results=max_urls,include_raw_content=True)
    return normalize_search(response,max_urls)

def validate_records(records):
    if not isinstance(records,list):raise ValueError("records must be a list")
    seen=set()
    for row in records:
        if not isinstance(row,dict) or any(not isinstance(row.get(k),str) or not row[k].strip() for k in ('id','source','text')):
            raise ValueError("invalid evidence record")
        if row['id'] in seen:raise ValueError("duplicate evidence ID")
        seen.add(row['id'])

def valid_answer(answer,docs):
    if not isinstance(answer,dict) or set(answer)!={'text','citations'}:return False
    citations=answer['citations']
    return (isinstance(answer['text'],str) and bool(answer['text'].strip())
            and isinstance(citations,list) and bool(citations)
            and all(isinstance(c,str) for c in citations)
            and len(citations)==len(set(citations))
            and set(citations)<={d['id'] for d in docs})

### TODO: retrieve
Return up to k copied records ranked by overlap of lowercase word tokens. Exclude zero-overlap records; resolve ties by input order. Validate nonnegative integer k. This is a lexical teaching baseline, not semantic retrieval.

In [3]:
def retrieve(query,records,k=3):
    if type(k) is not int or k<0:raise ValueError("invalid k")
    if not isinstance(query,str) or not query.strip():raise ValueError("invalid query")
    validate_records(records)
    raise NotImplementedError("TODO 4.1a: tokenize the query into q")
    scored=((len(q & set(re.findall(r"\w+",r["text"].lower()))),-i,r) for i,r in enumerate(records))
    raise NotImplementedError("TODO 4.1b: return copied positive-overlap top-k records")

### TODO: corrective_rag
Implement bounded retrieval, grading, rewrite/search repair, and abstention. Grade against the original question. Preserve source records and input nonmutation. Validate records and structured answers with citations before accepting them. A citation must identify retrieved evidence; membership is not entailment. Follow the helper contracts and failure tests.

In [4]:
def corrective_rag(question,records,grader,rewriter,searcher,answerer,max_steps=3):
    if type(max_steps) is not int or not 1<=max_steps<=5:raise ValueError("max_steps must be 1..5")
    if not isinstance(question,str) or not question.strip():raise ValueError("invalid question")
    validate_records(records)
    corpus={r['id']:dict(r) for r in records};query=question;trace=[]
    for step in range(max_steps):
        docs=retrieve(query,list(corpus.values()))
        trace.append({"query":query,"retrieved_ids":[d['id'] for d in docs]})
        if docs:
            raise NotImplementedError("TODO 4.2a: grade against the ORIGINAL question; save decision")
            if type(decision) is not bool:raise ValueError("grader must return bool")
            if decision:
                answer=answerer(question,docs)
                raise NotImplementedError("TODO 4.2b: reject invalid structured answers before the answered return")
                if False:
                    return {"answer":None,"trace":trace,"status":"invalid_answer"}
                return {"answer":answer,"trace":trace,"status":"answered"}
        if step+1<max_steps:
            query=rewriter(question)
            if not isinstance(query,str) or not query.strip():raise ValueError("invalid rewritten query")
            raise NotImplementedError("TODO 4.2c: search and validate additions before merging")
            for r in additions:
                if r['id'] in corpus and corpus[r['id']]!=r:raise ValueError("conflicting evidence ID")
                corpus[r['id']]=dict(r)
    return {"answer":"Insufficient evidence.","trace":trace,"status":"abstained"}

## Checks
Visible checks are examples, not a complete grader. Add two normal and three failure cases.

In [5]:
if RUN_EXERCISES:
    class FakeSearch:
        def __init__(self,**kwargs): assert kwargs=={"api_key":"offline-placeholder"}
        def search(self,**kwargs):
            assert kwargs["include_raw_content"] is True
            return {"results":[{"url":"https://example.org/orion","raw_content":"Orion battery lifetime is 8 hours."}]}
    records=search_and_scrape("Orion battery", "offline-placeholder", FakeSearch)
    assert records[0]["source"]=="https://example.org/orion"
    result=corrective_rag("Orion battery",[],
        grader=lambda q,docs:any("8 hours" in d["text"] for d in docs),
        rewriter=lambda q:q,
        searcher=lambda q:records,
        answerer=lambda q,docs:{"text":docs[0]["text"],"citations":[docs[0]["id"]]})
    assert result["status"]=="answered" and len(result["trace"])==2
    print(result)

In [6]:
if RUN_EXERCISES:
    for response in [None,{}, {"results":None},{"results":[{},None,{"url":"https://example.org","raw_context":"wrong field"}]}]:
        assert normalize_search(response)==[]
    assert normalize_search({"results":[{"url":"https://example.org/x","raw_content":"abc"}]*3})==[{"id":"https://example.org/x","source":"https://example.org/x","text":"abc"}]
    r=corrective_rag("unknown",[],lambda q,d:False,lambda q:q,lambda q:[],lambda q,d:"bad",max_steps=2)
    assert r["status"]=="abstained" and len(r["trace"])==2
    print("Foundation checks passed")

## Submission and rubric
Submit your implementation, test evidence, and a trace explanation. Implementation 50 points, added failure tests 25, trace and limitations 25. A default run with exercises disabled does not pass the lab. Do not import reference_support or instructor solutions to implement the tasks.